# LeetCode 2221 — Find Triangular Sum of an Array bằng Python

Link đề: https://leetcode.com/problems/find-triangular-sum-of-an-array/ · Độ khó: **Medium** · Chủ đề: Array, Math, Simulation, Combinatorics

Skeleton LeetCode:

```python
class Solution:
    def triangularSum(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô phỏng đúng quá trình "cộng hai phần tử kề nhau rồi lấy mod 10" — cả bằng mảng mới và **tại chỗ** (`O(1)` bộ nhớ).
2. Nhận ra **tam giác Pascal** ẩn bên trong: kết quả là `Σ C(n−1, i) · nums[i] (mod 10)`.
3. Hiểu tính chất **đồng dư** cho phép lấy `% 10` ở bất kỳ bước nào (hoặc chỉ ở cuối) mà không đổi kết quả.
4. Tính `C(n−1, i) mod 10` hiệu quả: với số nguyên lớn của Python, và bằng **định lý Lucas** cho mod 2, mod 5 rồi ghép bằng **định lý số dư Trung Hoa** (CRT) — vì 10 không nguyên tố nên **không** chia được theo mod 10.
5. Kiểm thử đối chiếu mọi cách trên dữ liệu ngẫu nhiên seed cố định.

## 2. Đề bài & ví dụ

Cho mảng `nums`, mỗi phần tử là một chữ số `0..9`. Lặp lại quá trình sau cho tới khi chỉ còn **một** phần tử:
- tạo mảng mới độ dài `n − 1` với `newNums[i] = (nums[i] + nums[i+1]) % 10`;
- thay `nums` bằng `newNums`.

Trả về phần tử cuối cùng (**tổng tam giác**).

**Ví dụ 1:** `[1,2,3,4,5]` → `8`:
```
1 2 3 4 5
 3 5 7 9
  8 2 6
   0 8
    8
```

**Ví dụ 2:** `[5]` → `5`.

**Ràng buộc:** `1 ≤ n ≤ 1000`, `0 ≤ nums[i] ≤ 9`.

In [1]:
import random
import time
from math import comb
from itertools import product
from typing import List

examples = [([1, 2, 3, 4, 5], 8), ([5], 5)]
for nums, exp in examples:
    print(nums, "-> kỳ vọng", exp)

[1, 2, 3, 4, 5] -> kỳ vọng 8
[5] -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1 Phép `%` và tính chất đồng dư

`(a + b) % 10 == ((a % 10) + (b % 10)) % 10`. Vì vậy lấy `% 10` ở **mỗi bước** hay chỉ ở **cuối** cho cùng kết quả — lấy ở mỗi bước chỉ để số không lớn lên.

In [2]:
a, b = 1234, 5678
print((a + b) % 10, ((a % 10) + (b % 10)) % 10)

2 2


### 3.2 Tam giác Pascal và `math.comb`

`C(n, k)` = số cách chọn `k` phần tử từ `n`; hàng `n` của tam giác Pascal là `C(n, 0), …, C(n, n)`, và mỗi số bằng tổng hai số phía trên: `C(n, k) = C(n−1, k−1) + C(n−1, k)`.

In [3]:
for n in range(6):
    print(" " * (6 - n) * 2, [comb(n, k) for k in range(n + 1)])

             [1]
           [1, 1]
         [1, 2, 1]
       [1, 3, 3, 1]
     [1, 4, 6, 4, 1]
   [1, 5, 10, 10, 5, 1]


### 3.3 Cập nhật mảng tại chỗ từ trái sang phải

`a[i] = a[i] + a[i+1]` với `i` **tăng dần**: khi tính `a[i]`, phần tử `a[i+1]` **chưa** bị sửa ⇒ đúng là giá trị cũ. (Duyệt giảm dần thì sai — mục 13.)

In [4]:
a = [1, 2, 3, 4, 5]
for i in range(len(a) - 1):
    a[i] = (a[i] + a[i + 1]) % 10
print("sau một vòng tại chỗ:", a[:-1], "(phần tử cuối bị bỏ qua)")

sau một vòng tại chỗ: [3, 5, 7, 9] (phần tử cuối bị bỏ qua)


### 3.4 Định lý Lucas (cho số nguyên tố `p`)

Viết `n`, `k` trong cơ số `p`: `n = (n_t … n_0)_p`, `k = (k_t … k_0)_p`. Khi đó `C(n, k) ≡ Π C(n_j, k_j) (mod p)` (với `C(a, b) = 0` nếu `b > a`). Ví dụ mod 2: `C(n, k)` lẻ ⇔ mọi bit của `k` đều là bit của `n` (`k & n == k`).

In [5]:
def lucas(n, k, p):
    res = 1
    while n or k:
        a, b = n % p, k % p
        if b > a: return 0
        res = res * comb(a, b) % p
        n //= p; k //= p
    return res

n = 10
print("C(10,k) mod 2:", [comb(n, k) % 2 for k in range(n + 1)], "| Lucas:", [lucas(n, k, 2) for k in range(n + 1)])
print("C(10,k) mod 5:", [comb(n, k) % 5 for k in range(n + 1)], "| Lucas:", [lucas(n, k, 5) for k in range(n + 1)])

C(10,k) mod 2: [1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1] | Lucas: [1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1]
C(10,k) mod 5: [1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 1] | Lucas: [1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 1]


### 3.5 Ghép mod 2 và mod 5 thành mod 10 (CRT)

Vì `10 = 2 · 5` và `gcd(2, 5) = 1`, mỗi cặp `(x mod 2, x mod 5)` ứng với **duy nhất** một `x mod 10`. Bảng tra 10 giá trị là đủ.

In [6]:
crt = {(x % 2, x % 5): x for x in range(10)}
print(crt)
print("x ≡ 1 (mod 2), x ≡ 3 (mod 5) -> x =", crt[(1, 3)])

{(0, 0): 0, (1, 1): 1, (0, 2): 2, (1, 3): 3, (0, 4): 4, (1, 0): 5, (0, 1): 6, (1, 2): 7, (0, 3): 8, (1, 4): 9}
x ≡ 1 (mod 2), x ≡ 3 (mod 5) -> x = 3


### 3.6 Nghịch đảo modulo `pow(x, -1, m)` — và vì sao mod 10 không dùng được

Chia cho `x` theo modulo `m` nghĩa là nhân với **nghịch đảo** `x⁻¹` (số `y` với `x·y ≡ 1 (mod m)`). Nghịch đảo tồn tại ⇔ `gcd(x, m) = 1`. Với `m = 10`, các số chẵn và các số chia hết cho 5 **không có** nghịch đảo ⇒ không thể tính `C(n, k) mod 10` bằng công thức có phép chia. Với số nguyên tố `p` (ví dụ `10⁹ + 7`), mọi `x` không chia hết cho `p` đều có nghịch đảo.

In [7]:
for x in range(1, 10):
    try:
        print(f"{x}^-1 mod 10 = {pow(x, -1, 10)}")
    except ValueError:
        print(f"{x}^-1 mod 10: KHÔNG tồn tại (gcd({x}, 10) = {__import__('math').gcd(x, 10)})")
print("3^-1 mod 1_000_000_007 =", pow(3, -1, 10**9 + 7))

1^-1 mod 10 = 1
2^-1 mod 10: KHÔNG tồn tại (gcd(2, 10) = 2)
3^-1 mod 10 = 7
4^-1 mod 10: KHÔNG tồn tại (gcd(4, 10) = 2)
5^-1 mod 10: KHÔNG tồn tại (gcd(5, 10) = 5)
6^-1 mod 10: KHÔNG tồn tại (gcd(6, 10) = 2)
7^-1 mod 10 = 3
8^-1 mod 10: KHÔNG tồn tại (gcd(8, 10) = 2)
9^-1 mod 10 = 9
3^-1 mod 1_000_000_007 = 333333336


## 4. Ý tưởng nền tảng / trực giác

Theo dõi **mỗi phần tử ban đầu đóng góp bao nhiêu lần** vào kết quả. Với `n = 5` (ví dụ 1), bỏ `% 10` để thấy rõ:

```
a     b     c     d     e
 a+b   b+c   c+d   d+e
  a+2b+c   b+2c+d   c+2d+e
   a+3b+3c+d   b+3c+3d+e
    a+4b+6c+4d+e
```

Hệ số `1, 4, 6, 4, 1` chính là hàng 4 của tam giác Pascal: `C(4, i)`. Vậy kết quả `= Σ C(n−1, i)·nums[i] (mod 10)`.

Ví dụ 1: `1·1 + 4·2 + 6·3 + 4·4 + 1·5 = 1 + 8 + 18 + 16 + 5 = 48` ⇒ `48 mod 10 = 8` ✓.

In [8]:
nums = [1, 2, 3, 4, 5]; n = len(nums)
coefs = [comb(n - 1, i) for i in range(n)]
print("hệ số:", coefs, "| tổng có trọng số:", sum(c * x for c, x in zip(coefs, nums)), "| mod 10:", sum(c * x for c, x in zip(coefs, nums)) % 10)

hệ số: [1, 4, 6, 4, 1] | tổng có trọng số: 48 | mod 10: 8


Thay vì tin vào hình tam giác ký hiệu, ta để máy **theo dõi vector hệ số**: mỗi phần tử của mảng là một vector độ dài `n` cho biết nó bằng tổng bao nhiêu lần `nums[0], nums[1], …`. Ban đầu phần tử `i` là vector đơn vị `e_i`; mỗi vòng cộng hai vector kề nhau. Sau `n − 1` vòng, vector còn lại chính là hàng Pascal.

In [9]:
def coefficient_vectors(n):
    vecs = [[1 if j == i else 0 for j in range(n)] for i in range(n)]
    while len(vecs) > 1:
        vecs = [[a + b for a, b in zip(vecs[i], vecs[i + 1])] for i in range(len(vecs) - 1)]
    return vecs[0]

for n in range(1, 8):
    v = coefficient_vectors(n)
    assert v == [comb(n - 1, i) for i in range(n)]
    print(f"n={n}: hệ số {v}")

n=1: hệ số [1]
n=2: hệ số [1, 1]
n=3: hệ số [1, 2, 1]
n=4: hệ số [1, 3, 3, 1]
n=5: hệ số [1, 4, 6, 4, 1]
n=6: hệ số [1, 5, 10, 10, 5, 1]
n=7: hệ số [1, 6, 15, 20, 15, 6, 1]


## 5. Cách 1 — Brute Force / baseline: mô phỏng bằng mảng mới

**Ý tưởng:** làm đúng như đề, tạo `newNums` ở mỗi vòng.

In [10]:
def tri_simulate(nums):
    cur = list(nums)
    while len(cur) > 1:
        cur = [(cur[i] + cur[i + 1]) % 10 for i in range(len(cur) - 1)]
    return cur[0]

for nums, exp in examples:
    got = tri_simulate(nums); print(nums, "->", got); assert got == exp

[1, 2, 3, 4, 5] -> 8
[5] -> 5


**Độ phức tạp:** `(n−1) + (n−2) + … + 1 = n(n−1)/2` phép cộng ⇒ `O(n²)`; với `n = 1000` khoảng `5·10⁵` — **chạy được** (đây cũng là lời giải được chấp nhận). Space `O(n)` mỗi vòng (cấp phát mảng mới).

**Có thể làm tốt hơn không?** Về bộ nhớ: cập nhật tại chỗ. Về thời gian: công thức Pascal cho `O(n)` phép nhân (số lớn) hoặc `O(n log n)` với Lucas.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tại chỗ):** ở mỗi vòng, `a[i] ← (a[i] + a[i+1]) % 10` với `i` tăng dần, rồi coi như mảng ngắn đi 1. Không cần mảng mới.

**Insight 2 (Pascal):** bằng quy nạp, sau `t` vòng phần tử thứ `i` là `Σ_j C(t, j)·nums[i+j]`. Sau `n−1` vòng: `Σ_j C(n−1, j)·nums[j]`.

**Insight 3 (mod 10 không chia được):** muốn tính `C(n−1, i) mod 10` dần bằng `C(n−1, i) = C(n−1, i−1)·(n−i)/i`, ta cần **chia** cho `i` — mod 10 không có nghịch đảo cho `i` chẵn hoặc chia hết cho 5. Hai lối thoát:
- dùng **số nguyên lớn** của Python (tính chính xác rồi mới `% 10`);
- tính `mod 2` và `mod 5` bằng **Lucas** (2, 5 nguyên tố) rồi ghép bằng **CRT**.

**Quy tắc "nếu… thì…" (Lucas):** **nếu** một chữ số cơ số `p` của `i` lớn hơn chữ số tương ứng của `n−1` **thì** `C(n−1, i) ≡ 0 (mod p)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng tại chỗ (bản nộp chính)

In [11]:
def tri_inplace(nums):
    a = list(nums)                   # không sửa input của người gọi
    for length in range(len(a), 1, -1):
        for i in range(length - 1):
            a[i] = (a[i] + a[i + 1]) % 10
    return a[0]

for nums, exp in examples:
    assert tri_inplace(nums) == exp
print([tri_inplace(nums) for nums, _ in examples])

[8, 5]


**Độ phức tạp Cách 2:** `O(n²)` thời gian, `O(1)` bộ nhớ thêm (ngoài bản sao để không phá input; trên LeetCode có thể sửa thẳng `nums`).

### 7.2 Cách 3 — Hệ số Pascal bằng `math.comb`

In [12]:
def tri_comb(nums):
    n = len(nums)
    return sum(comb(n - 1, i) * x for i, x in enumerate(nums)) % 10

for nums, exp in examples:
    assert tri_comb(nums) == exp
print([tri_comb(nums) for nums, _ in examples])

[8, 5]


**Độ phức tạp Cách 3:** `n` lần gọi `comb` trên số lớn (`C(999, 500)` có ~300 chữ số) — vẫn nhanh với `n = 1000`.

### 7.3 Cách 4 — Hàng Pascal bằng truy hồi số lớn

`C(m, i) = C(m, i−1) · (m − i + 1) / i` — với **số nguyên Python chính xác**, phép chia luôn chia hết. Chỉ lấy `% 10` khi nhân với `nums[i]`.

In [13]:
def tri_pascal_row(nums):
    m = len(nums) - 1
    c, total = 1, 0
    for i, x in enumerate(nums):
        total += c * x
        c = c * (m - i) // (i + 1)          # C(m, i+1), chia hết chính xác
    return total % 10

for nums, exp in examples:
    assert tri_pascal_row(nums) == exp
print([tri_pascal_row(nums) for nums, _ in examples])

[8, 5]


**Độ phức tạp Cách 4:** `O(n)` phép nhân/chia số lớn (mỗi số `O(n)` chữ số) ⇒ `O(n²)` thao tác chữ số trong trường hợp xấu nhất, nhưng thực tế rất nhanh.

### 7.4 Cách 5 — Lucas (mod 2, mod 5) + CRT, không cần số lớn

In [14]:
CRT10 = {(x % 2, x % 5): x for x in range(10)}

def comb_mod10(m, k):
    return CRT10[(lucas(m, k, 2), lucas(m, k, 5))]

def tri_lucas(nums):
    m = len(nums) - 1
    return sum(comb_mod10(m, i) * x for i, x in enumerate(nums)) % 10

for nums, exp in examples:
    assert tri_lucas(nums) == exp
print([tri_lucas(nums) for nums, _ in examples])

[8, 5]


**Độ phức tạp Cách 5:** mỗi hệ số `O(log₂ n + log₅ n)` ⇒ `O(n log n)`, chỉ dùng số nhỏ. Đây là cách duy nhất còn nhanh nếu `n` lên tới `10⁶` hay `10⁷`.

## 8. TRACE từng bước

Trace Cách 2 (tại chỗ) trên ví dụ 1:

| Vòng | Mảng (phần còn hiệu lực) |
|---|---|
| bắt đầu | 1 2 3 4 5 |
| 1 | 3 5 7 9 |
| 2 | 8 2 6 |
| 3 | 0 8 |
| 4 | **8** |

In [15]:
def trace_inplace(nums):
    a = list(nums)
    print("bắt đầu:", a)
    for r, length in enumerate(range(len(a), 1, -1), 1):
        for i in range(length - 1):
            a[i] = (a[i] + a[i + 1]) % 10
        print(f"vòng {r}: {a[:length - 1]}   (bộ nhớ thật: {a})")
    return a[0]

print("Đáp án:", trace_inplace([1, 2, 3, 4, 5]))

bắt đầu: [1, 2, 3, 4, 5]
vòng 1: [3, 5, 7, 9]   (bộ nhớ thật: [3, 5, 7, 9, 5])
vòng 2: [8, 2, 6]   (bộ nhớ thật: [8, 2, 6, 9, 5])
vòng 3: [0, 8]   (bộ nhớ thật: [0, 8, 6, 9, 5])
vòng 4: [8]   (bộ nhớ thật: [8, 8, 6, 9, 5])
Đáp án: 8


In [16]:
def trace_coefficients(nums):
    m = len(nums) - 1; total = 0
    for i, x in enumerate(nums):
        c = comb(m, i); c10 = comb_mod10(m, i); total += c10 * x
        print(f"i={i}: C({m},{i}) = {c} ≡ {c10} (mod 10) [mod2={lucas(m, i, 2)}, mod5={lucas(m, i, 5)}]  × nums[i]={x} -> tổng {total}")
    print("=> tổng mod 10 =", total % 10)

trace_coefficients([1, 2, 3, 4, 5])

i=0: C(4,0) = 1 ≡ 1 (mod 10) [mod2=1, mod5=1]  × nums[i]=1 -> tổng 1
i=1: C(4,1) = 4 ≡ 4 (mod 10) [mod2=0, mod5=4]  × nums[i]=2 -> tổng 9
i=2: C(4,2) = 6 ≡ 6 (mod 10) [mod2=0, mod5=1]  × nums[i]=3 -> tổng 27
i=3: C(4,3) = 4 ≡ 4 (mod 10) [mod2=0, mod5=4]  × nums[i]=4 -> tổng 43
i=4: C(4,4) = 1 ≡ 1 (mod 10) [mod2=1, mod5=1]  × nums[i]=5 -> tổng 48
=> tổng mod 10 = 8


In [17]:
def draw_triangle(nums):
    cur = list(nums); rows = [cur]
    while len(cur) > 1:
        cur = [(cur[i] + cur[i + 1]) % 10 for i in range(len(cur) - 1)]; rows.append(cur)
    w = len(nums)
    for r, row in enumerate(rows):
        print(" " * r + " ".join(map(str, row)))

draw_triangle([1, 2, 3, 4, 5])

1 2 3 4 5
 3 5 7 9
  8 2 6
   0 8
    8


**Họa tiết thú vị:** in tam giác Pascal mod 10 (thay số 0 bằng dấu chấm) cho 17 hàng đầu — các số 0 tạo thành những "lỗ" tam giác giống tam giác Sierpinski, phản ánh định lý Lucas (mod 2 và mod 5). Nhiều hệ số bằng 0 nghĩa là nhiều phần tử của `nums` **không ảnh hưởng** tới kết quả.

In [18]:
for m in range(17):
    row = [comb_mod10(m, k) for k in range(m + 1)]
    print(f"{m:2d} " + " " * (16 - m) + " ".join("." if v == 0 else str(v) for v in row))

 0                 1
 1                1 1
 2               1 2 1
 3              1 3 3 1
 4             1 4 6 4 1
 5            1 5 . . 5 1
 6           1 6 5 . 5 6 1
 7          1 7 1 5 5 1 7 1
 8         1 8 8 6 . 6 8 8 1
 9        1 9 6 4 6 6 4 6 9 1
10       1 . 5 . . 2 . . 5 . 1
11      1 1 5 5 . 2 2 . 5 5 1 1
12     1 2 6 . 5 2 4 2 5 . 6 2 1
13    1 3 8 6 5 7 6 6 7 5 6 8 3 1
14   1 4 1 4 1 2 3 2 3 2 1 4 1 4 1
15  1 5 5 5 5 3 5 5 5 5 3 5 5 5 5 1
16 1 6 . . . 8 8 . . . 8 8 . . . 6 1


## 9. Code nộp LeetCode

In [19]:
class Solution:
    def triangularSum(self, nums: List[int]) -> int:
        n = len(nums)
        for length in range(n, 1, -1):              # mỗi vòng mảng ngắn đi 1
            for i in range(length - 1):             # i tăng dần: nums[i+1] còn là giá trị cũ
                nums[i] = (nums[i] + nums[i + 1]) % 10
        return nums[0]

for nums, exp in examples:
    assert Solution().triangularSum(list(nums)) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Tại chỗ (Cách 2).** Ở vòng có độ dài `L`, ta muốn `b[i] = a[i] + a[i+1]` cho `i = 0..L−2`. Khi gán `a[i]` (theo `i` tăng), vị trí `a[i+1]` chưa bị gán trong vòng này ⇒ còn giá trị cũ ⇒ `a[i]` mới đúng bằng `b[i]`. Phần tử `a[L−1]` không còn dùng.

**Công thức Pascal (Cách 3–5).** Quy nạp theo số vòng `t`: phần tử vị trí `i` sau `t` vòng là `S_t(i) = Σ_{j=0}^{t} C(t, j)·nums[i+j]`. Với `t = 0` hiển nhiên. Bước `t → t+1`: `S_{t+1}(i) = S_t(i) + S_t(i+1) = Σ_j [C(t, j) + C(t, j−1)]·nums[i+j] = Σ_j C(t+1, j)·nums[i+j]` (đẳng thức Pascal). Sau `n − 1` vòng: `Σ_j C(n−1, j)·nums[j]`. Lấy `% 10` ở mọi bước không đổi kết quả cuối (mục 3.1).

**Lucas + CRT.** Lucas đúng cho mọi số nguyên tố `p`; `C(m, k) mod 10` được xác định duy nhất bởi `(mod 2, mod 5)` theo CRT.

In [20]:
# Kiểm chứng: hệ số Pascal mod 10 qua Lucas+CRT khớp comb(...) % 10 với mọi m <= 300
bad = sum(1 for m in range(301) for k in range(m + 1) if comb_mod10(m, k) != comb(m, k) % 10)
print("Số cặp (m, k) với m <= 300 mà Lucas+CRT sai:", bad)

Số cặp (m, k) với m <= 300 mà Lucas+CRT sai: 0


**Tính tay một hệ số bằng Lucas.** `C(10, 5) = 252 ≡ 2 (mod 10)`. Kiểm bằng Lucas:
- mod 2: `10 = 1010₂`, `5 = 0101₂`; bit 0 của `5` là 1 nhưng bit 0 của `10` là 0 ⇒ có `C(0, 1) = 0` ⇒ `C(10, 5) ≡ 0 (mod 2)`.
- mod 5: `10 = 20₅`, `5 = 10₅` ⇒ `C(2, 1)·C(0, 0) = 2` ⇒ `≡ 2 (mod 5)`.
- CRT: số `x ∈ [0, 9]` chẵn và `≡ 2 (mod 5)` là `x = 2` ✓.

In [21]:
print("C(10,5) =", comb(10, 5), "| mod 2 (Lucas):", lucas(10, 5, 2), "| mod 5 (Lucas):", lucas(10, 5, 5),
      "| CRT ->", CRT10[(lucas(10, 5, 2), lucas(10, 5, 5))])
print("C(999, 499) có", len(str(comb(999, 499))), "chữ số — Cách 3/4 phải làm việc với số lớn như vậy")

C(10,5) = 252 | mod 2 (Lucas): 0 | mod 5 (Lucas): 2 | CRT -> 2
C(999, 499) có 300 chữ số — Cách 3/4 phải làm việc với số lớn như vậy


## 11. Phân tích Time / Space

Gọi `n = len(nums) ≤ 1000`.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng mảng mới | `O(n²)` | `O(n)` mỗi vòng |
| 2. Tại chỗ | `O(n²)` | `O(1)` |
| 3. `math.comb` | `n` lần `comb` số lớn | `O(n)` chữ số |
| 4. Hàng Pascal số lớn | `O(n)` phép tính số lớn | `O(n)` chữ số |
| 5. Lucas + CRT | `O(n log n)` | `O(1)` |

In [22]:
rng = random.Random(2221)
for n in (1000, 5000):
    a = [rng.randint(0, 9) for _ in range(n)]
    row = []
    for name, fn in [("simulate", tri_simulate), ("inplace", tri_inplace), ("comb", tri_comb),
                     ("pascal_row", tri_pascal_row), ("lucas", tri_lucas)]:
        t = time.perf_counter(); r = fn(a); row.append(f"{name}={r} ({time.perf_counter() - t:.3f}s)")
    print(f"n={n}: " + ", ".join(row))
big = [rng.randint(0, 9) for _ in range(200000)]
t = time.perf_counter(); r = tri_lucas(big); print(f"n=200000 chỉ Lucas: {r} ({time.perf_counter() - t:.3f}s)")

n=1000: simulate=5 (0.016s), inplace=5 (0.016s), comb=5 (0.010s), pascal_row=5 (0.000s), lucas=5 (0.001s)


n=5000: simulate=5 (0.435s), inplace=5 (0.428s), comb=5 (0.818s), pascal_row=5 (0.005s), lucas=5 (0.005s)


n=200000 chỉ Lucas: 4 (0.261s)


## 12. Bộ test

- Ví dụ đề.
- Biên: `n = 1`; `n = 2`; toàn 0; toàn 9; `n` là lũy thừa của 2 hoặc 5 (nhiều hệ số Pascal ≡ 0).
- **Vét cạn** mọi mảng độ dài `1..5` trên `{0..9}` giới hạn (dùng `{0, 3, 7, 9}` cho nhanh) và ngẫu nhiên tới `n = 1000` (seed cố định).

In [23]:
fns = {"simulate": tri_simulate, "inplace": tri_inplace, "comb": tri_comb, "pascal_row": tri_pascal_row,
       "lucas": tri_lucas, "Solution": lambda a: Solution().triangularSum(list(a))}
edge = [([0], 0), ([9, 9], 8), ([0] * 7, 0), ([9] * 4, (9 * 8) % 10), ([1] + [0] * 7, 1), ([1, 0, 0, 0, 0, 0, 0, 1], 2)]
for nums, exp in examples + edge:
    for name, fn in fns.items():
        assert fn(nums) == exp, (name, nums, fn(nums))
cnt = 0
for n in range(1, 7):
    for tup in product((0, 3, 7, 9), repeat=n):
        want = tri_simulate(tup)
        for name, fn in fns.items():
            assert fn(list(tup)) == want, (name, tup)
        cnt += 1
print(f"Ví dụ + biên + vét cạn {cnt} mảng: OK")

Ví dụ + biên + vét cạn 5460 mảng: OK


In [24]:
rng = random.Random(42)
for _ in range(30):
    a = [rng.randint(0, 9) for _ in range(rng.randint(1, 1000))]
    want = tri_simulate(a)
    for name, fn in fns.items():
        assert fn(a) == want, name
print("30 test ngẫu nhiên (n <= 1000): mọi cách khớp mô phỏng")

30 test ngẫu nhiên (n <= 1000): mọi cách khớp mô phỏng


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Cập nhật tại chỗ **từ phải sang trái**

Khi duyệt `i` giảm dần, `a[i+1]` đã bị ghi đè bởi giá trị mới ⇒ sai.

In [25]:
def wrong_overwrite(nums):
    a = list(nums)
    for length in range(len(a), 1, -1):
        for i in range(length - 2, -1, -1):
            a[i] = (a[i] + a[i + 1]) % 10               # SAI: a[i+1] đã là giá trị MỚI
    return a[0]

print("sai:", wrong_overwrite([1, 2, 3, 4, 5]), "| đúng:", 8)

sai: 2 | đúng: 8


In [26]:
print("sửa (i tăng dần):", tri_inplace([1, 2, 3, 4, 5]))

sửa (i tăng dần): 8


### Lỗi 2 — Tính hệ số Pascal mod 10 bằng "chia" theo mod 10

`C(m, i+1) = C(m, i)·(m−i)/(i+1)` — nếu lấy `% 10` trước rồi mới chia (`//`), phép chia không còn chính xác.

In [27]:
def wrong_mod_division(nums):
    m = len(nums) - 1; c = 1; total = 0
    for i, x in enumerate(nums):
        total += c * x
        c = (c * (m - i) // (i + 1)) % 10              # SAI: c đã bị lấy mod nên phép chia sai
    return total % 10

bad = next(list(t) for n in range(2, 9) for t in product(range(1, 4), repeat=n) if wrong_mod_division(list(t)) != tri_simulate(t))
print("phản ví dụ:", bad, "| sai:", wrong_mod_division(bad), "| đúng:", tri_simulate(bad))

phản ví dụ: [1, 1, 1, 1, 1, 1] | sai: 6 | đúng: 2


In [28]:
print("sửa (số lớn chính xác hoặc Lucas+CRT):", tri_pascal_row(bad), tri_lucas(bad))

sửa (số lớn chính xác hoặc Lucas+CRT): 2 2


### Lỗi 3 — Dùng hàng Pascal sai: `C(n, i)` thay vì `C(n−1, i)`

In [29]:
def wrong_row(nums):
    n = len(nums)
    return sum(comb(n, i) * x for i, x in enumerate(nums)) % 10    # SAI: phải là n - 1

print("sai:", wrong_row([1, 2, 3, 4, 5]), "| đúng:", 8)

sai: 6 | đúng: 8


In [30]:
print("sửa:", tri_comb([1, 2, 3, 4, 5]))

sửa: 8


### Lỗi 4 — Vòng lặp lệch một (thêm một vòng thừa / thiếu một vòng)

Số vòng phải đúng `n − 1`. Bản sai dưới đây dừng khi còn **2** phần tử và trả phần tử đầu.

In [31]:
def wrong_rounds(nums):
    cur = list(nums)
    while len(cur) > 2:                                  # SAI: phải là > 1
        cur = [(cur[i] + cur[i + 1]) % 10 for i in range(len(cur) - 1)]
    return cur[0]

print("sai:", wrong_rounds([1, 2, 3, 4, 5]), "| đúng:", 8)

sai: 0 | đúng: 8


In [32]:
print("sửa:", tri_simulate([1, 2, 3, 4, 5]))

sửa: 8


### Lỗi 5 — Lucas với `p = 10` (10 không phải số nguyên tố)

Định lý Lucas chỉ đúng với số nguyên tố. Áp dụng thẳng cho cơ số 10 sai ở nhiều hệ số.

In [33]:
bad_pairs = [(m, k) for m in range(30) for k in range(m + 1) if lucas(m, k, 10) != comb(m, k) % 10]
print("số cặp (m <= 29) sai khi dùng Lucas với p = 10:", len(bad_pairs), "| ví dụ:", bad_pairs[:3],
      [(comb(m, k) % 10, lucas(m, k, 10)) for m, k in bad_pairs[:3]])

số cặp (m <= 29) sai khi dùng Lucas với p = 10: 222 | ví dụ: [(10, 2), (10, 5), (10, 8)] [(5, 0), (2, 0), (5, 0)]


In [34]:
print("sửa (mod 2 & mod 5 rồi CRT):", [(comb_mod10(m, k), comb(m, k) % 10) for m, k in bad_pairs[:3]])

sửa (mod 2 & mod 5 rồi CRT): [(5, 5), (2, 2), (5, 5)]


### Lỗi 6 — Sửa thẳng mảng đầu vào khi người gọi còn dùng nó

Trên LeetCode, sửa `nums` tại chỗ là chấp nhận được. Nhưng trong code thật (hoặc khi test nhiều cách trên cùng một mảng) việc ghi đè làm hỏng dữ liệu của người gọi: lần gọi thứ hai cho kết quả khác.

In [35]:
data = [1, 2, 3, 4, 5]
r1 = Solution().triangularSum(data)          # sửa thẳng data
r2 = Solution().triangularSum(data)          # data đã bị thay đổi!
print("lần 1:", r1, "| lần 2 trên cùng list:", r2, "| data giờ là:", data)

lần 1: 8 | lần 2 trên cùng list: 7 | data giờ là: [7, 8, 9, 4, 5]


In [36]:
data = [1, 2, 3, 4, 5]
print("sửa (truyền bản sao hoặc dùng tri_inplace có list(nums)):", Solution().triangularSum(list(data)), Solution().triangularSum(list(data)), "| data:", data)

sửa (truyền bản sao hoặc dùng tri_inplace có list(nums)): 8 8 | data: [1, 2, 3, 4, 5]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Quá trình "cộng hai phần tử kề, mảng ngắn đi 1"** ⇒ tam giác Pascal ⇒ hệ số nhị thức `C(n−1, i)`.
- **Phép toán tuyến tính + mod** ⇒ lấy mod ở bất kỳ đâu.
- **Mod hợp số** (10, 6, …) ⇒ không chia được; tách thành các mod nguyên tố + CRT, dùng Lucas.
- Cập nhật tại chỗ: duyệt theo hướng sao cho giá trị cần đọc **chưa** bị ghi đè.
- `n ≤ 1000` ⇒ `O(n²)` mô phỏng là đủ; công thức chỉ cần khi `n` lớn.

## 15. Biến thể của bài

1. **Tam giác hiệu** `newNums[i] = nums[i+1] − nums[i]` (bài tập 5): hệ số `(−1)^{n−1−i} C(n−1, i)`.
2. **Mod số nguyên tố lớn** (ví dụ `10⁹+7`) với `n` lớn: dùng giai thừa + nghịch đảo modulo.
3. **LeetCode 118/119 — Pascal's Triangle:** sinh hàng tam giác Pascal.
4. **Phép XOR thay cho cộng:** kết quả là XOR các `nums[i]` có `C(n−1, i)` lẻ ⇔ `i & (n−1) == i` (Lucas mod 2).

Minh họa biến thể 4 (tự dựng, kiểm bằng mô phỏng):

In [37]:
def xor_triangle(nums):
    m = len(nums) - 1; res = 0
    for i, x in enumerate(nums):
        if i & m == i:                 # C(m, i) lẻ
            res ^= x
    return res

def xor_triangle_sim(nums):
    cur = list(nums)
    while len(cur) > 1:
        cur = [cur[i] ^ cur[i + 1] for i in range(len(cur) - 1)]
    return cur[0]

rng = random.Random(4)
for _ in range(300):
    a = [rng.randint(0, 255) for _ in range(rng.randint(1, 60))]
    assert xor_triangle(a) == xor_triangle_sim(a)
print("Biến thể XOR khớp mô phỏng; [1,2,3,4,5] ->", xor_triangle([1, 2, 3, 4, 5]))

Biến thể XOR khớp mô phỏng; [1,2,3,4,5] -> 4


Minh họa biến thể 2 (tự dựng) — **tổng tam giác theo modulo nguyên tố lớn** `P = 10⁹ + 7` với `n` lớn: tiền tính giai thừa và nghịch đảo giai thừa, `C(m, k) = m! · (k!)⁻¹ · ((m−k)!)⁻¹ mod P`. Kiểm bằng mô phỏng (mod `P`) cho `n` nhỏ và bằng `math.comb` cho `n` vừa.

In [38]:
P_MOD = 10**9 + 7

def tri_mod_prime(nums, p=P_MOD):
    m = len(nums) - 1
    fact = [1] * (m + 1)
    for i in range(1, m + 1): fact[i] = fact[i - 1] * i % p
    inv = [1] * (m + 1)
    inv[m] = pow(fact[m], -1, p)
    for i in range(m, 0, -1): inv[i - 1] = inv[i] * i % p
    return sum(fact[m] * inv[k] % p * inv[m - k] % p * x for k, x in enumerate(nums)) % p

def tri_mod_prime_sim(nums, p=P_MOD):
    cur = list(nums)
    while len(cur) > 1:
        cur = [(cur[i] + cur[i + 1]) % p for i in range(len(cur) - 1)]
    return cur[0]

rng = random.Random(7)
for _ in range(100):
    a = [rng.randint(0, 10**9) for _ in range(rng.randint(1, 60))]
    assert tri_mod_prime(a) == tri_mod_prime_sim(a)
a = [rng.randint(0, 9) for _ in range(3000)]
assert tri_mod_prime(a) == sum(comb(2999, k) * x for k, x in enumerate(a)) % P_MOD
print("Tổng tam giác mod 1e9+7 bằng giai thừa + nghịch đảo khớp mô phỏng và math.comb")

Tổng tam giác mod 1e9+7 bằng giai thừa + nghịch đảo khớp mô phỏng và math.comb


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng mảng mới | đúng đề | `O(n²)` | `O(n)` | dễ hiểu |
| 2. Tại chỗ | ghi đè từ trái sang | `O(n²)` | `O(1)` | **nộp bài** |
| 3. `math.comb` | hệ số Pascal | `O(n·cost(comb))` | số lớn | ngắn gọn |
| 4. Hàng Pascal số lớn | truy hồi chính xác | `O(n)` phép số lớn | số lớn | nhanh với `n` vừa |
| 5. Lucas + CRT | hệ số mod 10 trực tiếp | `O(n log n)` | `O(1)` | `n` rất lớn |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `triangularSum([9, 9, 9])`.
2. **Tính tay:** `triangularSum([2, 0, 0, 0, 0, 0, 0, 0, 3])` bằng hệ số Pascal (gợi ý: `n − 1 = 8`).
3. **Điền TODO:** `pascal_row_mod10(m)` trả về `[C(m, 0) % 10, …, C(m, m) % 10]` mà không dùng số lớn.
4. **Tự viết lại:** viết lại Cách 2 không nhìn code.
5. **Mini challenge:** `difference_triangle(nums)` — mỗi vòng `newNums[i] = (nums[i+1] − nums[i]) % 10`. Tìm kết quả bằng công thức và kiểm bằng mô phỏng.

In [39]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def pascal_row_mod10(m):
    # TODO: dùng comb_mod10 (Lucas + CRT)
    pass

In [40]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_triangular_sum(nums):
    # TODO
    pass

In [41]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def difference_triangle(nums):
    # TODO: hệ số (-1)^(n-1-i) * C(n-1, i)
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Hệ số `1, 2, 1`. (2) `9 + 18 + 9 = 36`. (3) `6`.

**Bài 2.** (1) `C(8, 0) = 1`, `C(8, 8) = 1`. (2) Các phần tử giữa bằng 0. (3) `2·1 + 3·1 = 5`.

**Bài 3.** (1) Duyệt `k = 0..m`. (2) `comb_mod10(m, k)`. (3) So với `comb(m, k) % 10`.

**Bài 4.** (1) Vòng ngoài độ dài giảm dần. (2) Vòng trong `i` tăng dần. (3) Trả `nums[0]`.

**Bài 5.** (1) Sau một vòng: `b[i] = a[i+1] − a[i]`. (2) Quy nạp: hệ số của `nums[i]` là `(−1)^{n−1−i} C(n−1, i)`. (3) Lấy `% 10` (Python `%` cho kết quả không âm).

## 19. Đáp án tham khảo

In [42]:
# Bài 1, 2
assert Solution().triangularSum([9, 9, 9]) == 6
assert Solution().triangularSum([2, 0, 0, 0, 0, 0, 0, 0, 3]) == 5
print("Bài 1: 6 | Bài 2: 5")
trace_coefficients([2, 0, 0, 0, 0, 0, 0, 0, 3])

Bài 1: 6 | Bài 2: 5
i=0: C(8,0) = 1 ≡ 1 (mod 10) [mod2=1, mod5=1]  × nums[i]=2 -> tổng 2
i=1: C(8,1) = 8 ≡ 8 (mod 10) [mod2=0, mod5=3]  × nums[i]=0 -> tổng 2
i=2: C(8,2) = 28 ≡ 8 (mod 10) [mod2=0, mod5=3]  × nums[i]=0 -> tổng 2
i=3: C(8,3) = 56 ≡ 6 (mod 10) [mod2=0, mod5=1]  × nums[i]=0 -> tổng 2
i=4: C(8,4) = 70 ≡ 0 (mod 10) [mod2=0, mod5=0]  × nums[i]=0 -> tổng 2
i=5: C(8,5) = 56 ≡ 6 (mod 10) [mod2=0, mod5=1]  × nums[i]=0 -> tổng 2
i=6: C(8,6) = 28 ≡ 8 (mod 10) [mod2=0, mod5=3]  × nums[i]=0 -> tổng 2
i=7: C(8,7) = 8 ≡ 8 (mod 10) [mod2=0, mod5=3]  × nums[i]=0 -> tổng 2
i=8: C(8,8) = 1 ≡ 1 (mod 10) [mod2=1, mod5=1]  × nums[i]=3 -> tổng 5
=> tổng mod 10 = 5


In [43]:
# Bài 3, 4
def pascal_row_mod10_ans(m):
    return [comb_mod10(m, k) for k in range(m + 1)]

my_triangular_sum_ans = tri_inplace
for m in range(60):
    assert pascal_row_mod10_ans(m) == [comb(m, k) % 10 for k in range(m + 1)]
rng = random.Random(3)
for _ in range(100):
    a = [rng.randint(0, 9) for _ in range(rng.randint(1, 200))]
    assert my_triangular_sum_ans(a) == tri_simulate(a)
print("Bài 3, 4: OK; hàng 10 mod 10:", pascal_row_mod10_ans(10))

Bài 3, 4: OK; hàng 10 mod 10: [1, 0, 5, 0, 0, 2, 0, 0, 5, 0, 1]


In [44]:
# Bài 5
def difference_triangle_ans(nums):
    m = len(nums) - 1
    return sum((-1) ** (m - i) * comb_mod10(m, i) * x for i, x in enumerate(nums)) % 10

def difference_triangle_sim(nums):
    cur = list(nums)
    while len(cur) > 1:
        cur = [(cur[i + 1] - cur[i]) % 10 for i in range(len(cur) - 1)]
    return cur[0]

rng = random.Random(5)
for _ in range(300):
    a = [rng.randint(0, 9) for _ in range(rng.randint(1, 80))]
    assert difference_triangle_ans(a) == difference_triangle_sim(a)
print("Bài 5: OK; [1,2,3,4,5] ->", difference_triangle_ans([1, 2, 3, 4, 5]), "(hiệu bậc 4 của dãy tuyến tính = 0)")

Bài 5: OK; [1,2,3,4,5] -> 0 (hiệu bậc 4 của dãy tuyến tính = 0)


**Hệ quả của Lucas mod 2:** số hệ số **lẻ** trong hàng `m` của tam giác Pascal là `2^{popcount(m)}` (mỗi bit 1 của `m` cho 2 lựa chọn bit của `k`). Ví dụ hàng `m = 8 = 1000₂` chỉ có 2 hệ số lẻ (`C(8,0)` và `C(8,8)`) — nên trong biến thể XOR, chỉ phần tử đầu và cuối quyết định kết quả khi `n = 9`.

In [45]:
for m in (7, 8, 10, 15, 16):
    odd = sum(comb(m, k) % 2 for k in range(m + 1))
    print(f"m={m:2d} ({m:05b}): số hệ số lẻ = {odd} = 2^{bin(m).count('1')} = {2 ** bin(m).count('1')}")

m= 7 (00111): số hệ số lẻ = 8 = 2^3 = 8
m= 8 (01000): số hệ số lẻ = 2 = 2^1 = 2
m=10 (01010): số hệ số lẻ = 4 = 2^2 = 4
m=15 (01111): số hệ số lẻ = 16 = 2^4 = 16
m=16 (10000): số hệ số lẻ = 2 = 2^1 = 2


**Giải thích thêm cho Bài 5.** Tam giác hiệu giống tam giác tổng nhưng mỗi lần cộng phần tử **phải** và trừ phần tử **trái**; bằng quy nạp như mục 10, hệ số của `nums[i]` là `(−1)^{n−1−i}·C(n−1, i)` — chính là **sai phân bậc `n−1`**. Vì vậy với dãy là đa thức bậc `< n − 1` theo chỉ số (ví dụ `1, 2, 3, 4, 5` bậc 1), kết quả là 0. Ô dưới in tam giác hiệu của `[1, 4, 9, 16, 25]` (bình phương, bậc 2): hàng thứ 3 là hằng số 2, hàng 4 là 0.

In [46]:
cur = [1, 4, 9, 16, 25]
while cur:
    print(cur)
    cur = [cur[i + 1] - cur[i] for i in range(len(cur) - 1)]

[1, 4, 9, 16, 25]
[3, 5, 7, 9]
[2, 2, 2]
[0, 0]
[0]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template tại chỗ:**

```python
for length in range(n, 1, -1):
    for i in range(length - 1):          # i TĂNG dần
        a[i] = (a[i] + a[i + 1]) % 10
return a[0]
```

**Template công thức:** `Σ C(n−1, i)·nums[i] mod 10`, với `C mod 10` = CRT(Lucas mod 2, Lucas mod 5).

In [47]:
def template(nums):
    a = list(nums)
    for length in range(len(a), 1, -1):
        for i in range(length - 1):
            a[i] = (a[i] + a[i + 1]) % 10
    return a[0]
print([template(a) for a, _ in examples])

[8, 5]


**Checklist:**

- [ ] Đúng `n − 1` vòng (dừng khi còn 1 phần tử).
- [ ] Cập nhật tại chỗ với `i` **tăng** dần.
- [ ] Nếu dùng công thức: hàng `C(n−1, ·)`, không phải `C(n, ·)`.
- [ ] Không "chia theo mod 10"; dùng số lớn chính xác hoặc Lucas (2, 5) + CRT.
- [ ] Đã thử `n = 1`, `n = 2`, toàn 9.